# Phase 2 · BIO BiLSTM-CRF with our pretrained GQA/MHA

This experiment replaces independent all-character span classification with one coherent BIO sequence per deterministic word/punctuation window. It uses only our pretrained legal decoder checkpoints. The decoder remains causal; a small task-specific BiLSTM supplies right context before a constrained 25-tag CRF.

The audit found no overlapping or nested gold entities. Exact word-boundary representation covers 8,187/8,393 entities (97.55%). Boundary-ambiguous training words split the CRF loss into contiguous valid segments and are never trained as background. Validation retains all original gold entities and exact character offsets.


In [15]:
# 1 · Readable experiment configuration
from pathlib import Path
ACTION = "Train"  # Inspect | Overfit check | Train
PRESET = "gqa_pre"  # gqa_pre | mha_pre
COURSE_ROOT = Path("/content/drive/MyDrive/legal-llm-course")
TRAIN_DIR = COURSE_ROOT / "Phase_2_LNER/Train"
RUN_ROOT = COURSE_ROOT / "runs/phase2_lner/bio_bilstm_crf" / PRESET
BIO_HELPER = COURSE_ROOT / "bio_crf.py"
MAX_BPE, STRIDE_BPE = 512, 384
SEED, VALIDATION_FRACTION = 17, .15
MAX_EPOCHS, PATIENCE = 20, 4
BACKBONE_LR, HEAD_LR = 1e-5, 1e-3
WARMUP_UPDATES, CHECKPOINT_EVERY = 100, 500
LSTM_HIDDEN, INFERENCE_THRESHOLD = 256, None  # CRF decoding has no span threshold


In [2]:
# 2 · Mount Drive and load the existing bundle plus BIO helper
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, hashlib, json, importlib.util, subprocess, zipfile, random, math, time
BUNDLE_ZIP = COURSE_ROOT / "phase2_lner_gqa_repaired_bundle.zip"
LOCAL = Path('/content/phase2-bio-crf')
if LOCAL.exists(): shutil.rmtree(LOCAL)
if not BUNDLE_ZIP.is_file(): raise FileNotFoundError(f'Existing repaired bundle missing: {BUNDLE_ZIP}')
if not BIO_HELPER.is_file(): raise FileNotFoundError(f'BIO helper missing: {BIO_HELPER}')
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as archive: archive.extractall(LOCAL)
shutil.copy2(BIO_HELPER,LOCAL/'bio_crf.py')
sys.path.insert(0,str(LOCAL))
missing=[]
if importlib.util.find_spec('tokenizers') is None: missing.append('tokenizers==0.23.1')
if importlib.util.find_spec('huggingface_hub') is None: missing.append('huggingface_hub')
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
from tokenizers import Tokenizer
from legal_ner import *
from bio_crf import *
TOKENIZER_PATH=LOCAL/'tokenizer/tokenizer.json'
TOKENIZER_SHA256=hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
if TOKENIZER_SHA256!='906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab':
    raise ValueError('tokenizer identity mismatch')
TOKENIZER=Tokenizer.from_file(str(TOKENIZER_PATH))
print('Existing tokenizer/data helper and BIO helper loaded.')


Mounted at /content/drive
Existing tokenizer/data helper and BIO helper loaded.


In [3]:
# 3 · Download the pinned dataset files if needed, then audit exact pairs
from google.colab import userdata
training_files=ensure_training_files(COURSE_ROOT,userdata.get('HF_TOKEN'))
print('Training files ready:',[str(p.relative_to(COURSE_ROOT)) for p in training_files])
files=sorted(p for p in TRAIN_DIR.rglob('*') if p.is_file())
print('Files:',[(p.name,p.stat().st_size) for p in files])
labels_path=next((p for p in files if p.name.lower()=='labels.txt'),None)
if labels_path is None: raise FileNotFoundError('actual labels.txt not found')
load_labels(labels_path)
text_files={p.name[:-10]:p for p in files if p.name.endswith('-text.json')}
gold_files={p.name[:-10]:p for p in files if p.name.endswith('-gold.json')}
if not text_files or set(text_files)!=set(gold_files):
    raise ValueError(f'Expected paired official *-text.json/*-gold.json folds; text={sorted(text_files)}, gold={sorted(gold_files)}')
rows=[]
for fold in sorted(text_files):
    try: rows.extend(load_fold_pair(text_files[fold],gold_files[fold]))
    except Exception as e: raise ValueError(f'Could not join actual {fold} fold: {e}') from e
if len({r['id'] for r in rows})!=len(rows): raise ValueError('document ID appears in more than one fold')
report=audit(rows)
print(json.dumps(report,ensure_ascii=False,indent=2))
assert tuple(report['label_counts'])==LABELS
if report['substring_evidence_entities']:
    print('End-exclusive Python Unicode offsets verified against every provided entity_text value.')
else:
    print('WARNING: no entity_text evidence exists; offset convention is not independently proven by the annotations.')


Training files ready: ['Phase_2_LNER/Train/fold1-text.json', 'Phase_2_LNER/Train/fold1-gold.json', 'Phase_2_LNER/Train/fold2-text.json', 'Phase_2_LNER/Train/fold2-gold.json', 'Phase_2_LNER/Train/labels.txt']
Files: [('fold1-gold.json', 460581), ('fold1-text.json', 1106635), ('fold2-gold.json', 475489), ('fold2-text.json', 1124295), ('labels.txt', 71)]
{
  "documents": 70,
  "entities": 8393,
  "label_counts": {
    "APP": 407,
    "RESP": 381,
    "A.COUNSEL": 195,
    "R.COUNSEL": 184,
    "JUDGE": 245,
    "WIT": 187,
    "AUTH": 1003,
    "COURT": 1013,
    "STAT": 1260,
    "PREC": 1183,
    "DATE": 1575,
    "CASENO": 760
  },
  "unicode_documents": 29,
  "overlapping_pairs": 0,
  "nested_pairs": 0,
  "document_chars": {
    "0": 10443,
    "0.5": 26600,
    "0.9": 48800,
    "0.99": 106947,
    "1": 114044
  },
  "span_chars": {
    "0": 1,
    "0.5": 13,
    "0.9": 33,
    "0.99": 85,
    "1": 233
  },
  "substring_evidence_entities": 8393,
  "offset_convention": "verified on ev

In [4]:
# 4 · Preserve the 64/6 document split; deterministic windows never inspect gold
train_rows,val_rows=split_documents(rows,VALIDATION_FRACTION,SEED)
if (len(train_rows),len(val_rows))!=(64,6):
    raise ValueError(f'expected preserved 64/6 document split, got {len(train_rows)}/{len(val_rows)}')
assert {row['id'] for row in train_rows}.isdisjoint({row['id'] for row in val_rows})
train_windows=[window for row in train_rows for window in make_document_windows(row,TOKENIZER,MAX_BPE,STRIDE_BPE)]
val_windows=[window for row in val_rows for window in make_document_windows(row,TOKENIZER,MAX_BPE,STRIDE_BPE)]
aligned=ambiguous=0
for window in train_windows:
    _,segments,masked=targets_and_segments(window,LABELS)
    aligned+=sum(end-first for first,end in segments); ambiguous+=len(masked)
print({'train_documents':64,'validation_documents':6,'train_windows':len(train_windows),
       'validation_windows':len(val_windows),'supervised_train_word_positions':aligned,
       'ambiguous_train_word_occurrences_excluded_from_crf_segments':ambiguous})
print('Validation keeps every original gold span:',sum(len(row['entities']) for row in val_rows))


{'train_documents': 64, 'validation_documents': 6, 'train_windows': 1281, 'validation_windows': 111, 'supervised_train_word_positions': 542378, 'ambiguous_train_word_occurrences_excluded_from_crf_segments': 1950}
Validation keeps every original gold span: 781


## Backbone and task head

The snapshot resolver checks the exact architecture and tokenizer identity. GQA/MHA attention masks are unchanged. Each deterministic word uses its last overlapping decoder subtoken; the BiLSTM and CRF are new task layers.


In [17]:
# One-time reset before clean model-load; does not modify checkpoints or cell sources.
import random, torch
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("seeded Python/Torch/CUDA:", SEED, "CUDA available:", torch.cuda.is_available())

seeded Python/Torch/CUDA: 17 CUDA available: True


In [19]:
# 6 · Loss, exact validation, and optional one-window overfit check
def window_loss(window,device):
    ids=torch.tensor([window['input_ids']],dtype=torch.long,device=device)
    last=torch.tensor(window['word_last'],dtype=torch.long,device=device)
    targets,segments,ambiguous=targets_and_segments(window,LABELS); targets=targets.to(device)
    emissions=model.emissions(ids,last)
    losses=[model.crf.neg_log_likelihood(emissions[first:end],targets[first:end])
            for first,end in segments if end>first]
    if not losses: raise ValueError('window contains no contiguous supervised BIO segment')
    return torch.stack(losses).mean(),len(ambiguous)

@torch.inference_mode()
def validate(device):
    was_training=model.training; model.eval(); predictions=[]
    for window in val_windows:
        ids=torch.tensor([window['input_ids']],dtype=torch.long,device=device)
        last=torch.tensor(window['word_last'],dtype=torch.long,device=device)
        with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=(device=='cuda')):
            emissions=model.emissions(ids,last)
        predictions.extend(decode_entities(window,model.crf.decode(emissions.float()),model.tags))
    model.train(was_training)
    predicted=set(predictions)
    gold={(row['id'],a,b,label) for row in val_rows for a,b,label in row['entities']}
    per_label={}
    for label in LABELS:
        label_gold={item for item in gold if item[3]==label}; label_pred={item for item in predicted if item[3]==label}
        tp=len(label_gold&label_pred); precision=tp/len(label_pred) if label_pred else 0.; recall=tp/len(label_gold) if label_gold else 0.
        f1=2*precision*recall/(precision+recall) if precision+recall else 0.
        per_label[label]={'gold':len(label_gold),'predicted':len(label_pred),'tp':tp,'precision':precision,'recall':recall,'f1':f1}
    return per_label,sum(item['f1'] for item in per_label.values())/len(LABELS),len(predicted)

if ACTION=='Overfit check':
    if not torch.cuda.is_available(): raise RuntimeError('Overfit check requires CUDA')
    device='cuda'; model.to(device).train(); probe=train_windows[0]
    optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4)
    for update in range(20):
        optimizer.zero_grad(set_to_none=True); loss,masked=window_loss(probe,device); loss.backward(); optimizer.step()
        if update in (0,4,9,19): print('overfit update',update+1,'loss',float(loss.detach()),'masked',masked)
    print('Optional overfit check complete; no checkpoint was written.')
else: print('Optional overfit check skipped.')


Optional overfit check skipped.


In [20]:
# 7 · Full fine-tuning with robust latest/best recovery
def optimizer_groups(model):
    backbone=[]; head=[]; seen=set()
    for name,param in model.named_parameters(remove_duplicate=False):
        if not param.requires_grad or id(param) in seen: continue
        seen.add(id(param)); (backbone if name.startswith('backbone.') else head).append(param)
    return [{'params':backbone,'lr':BACKBONE_LR,'weight_decay':.01},
            {'params':head,'lr':HEAD_LR,'weight_decay':.01}]

def lr_multiplier(step,total):
    if step<WARMUP_UPDATES: return (step+1)/max(1,WARMUP_UPDATES)
    fraction=(step-WARMUP_UPDATES)/max(1,total-WARMUP_UPDATES)
    return .5*(1+math.cos(math.pi*min(fraction,1.)))

if ACTION=='Train':
    if not torch.cuda.is_available(): raise RuntimeError('Train requires CUDA; refusing CPU fallback')
    device='cuda'; model.to(device).train(); RUN_ROOT.mkdir(parents=True,exist_ok=True)
    optimizer=torch.optim.AdamW(optimizer_groups(model),betas=(.9,.999))
    total_updates=MAX_EPOCHS*len(train_windows)
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:lr_multiplier(step,total_updates))
    scaler=torch.amp.GradScaler('cuda'); latest=RUN_ROOT/'latest.pt'; best=RUN_ROOT/'best.pt'
    progress={'epoch':0,'next_position':0,'successful_updates':0,'overflow_retries':0,
              'best_val_macro_f1':float('-inf'),'best_epoch':-1,'epochs_without_improvement':0}
    if latest.exists():
        progress=resume_checkpoint(latest,model,optimizer,scheduler,scaler,IDENTITY,device)
        print('Resumed:',progress)
    started=time.monotonic()
    for epoch in range(int(progress['epoch']),MAX_EPOCHS):
        order=list(range(len(train_windows))); random.Random(SEED+epoch).shuffle(order)
        position=int(progress['next_position']) if epoch==int(progress['epoch']) else 0
        while position<len(order):
            optimizer.zero_grad(set_to_none=True); window=train_windows[order[position]]
            with torch.autocast(device_type='cuda',dtype=torch.float16): loss,masked=window_loss(window,device)
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            finite=all(torch.isfinite(param.grad).all().item() for param in model.parameters() if param.grad is not None)
            if not finite:
                scaler.step(optimizer); scaler.update(); progress['overflow_retries']+=1
                print(f'AMP overflow epoch={epoch+1} position={position}; retrying',flush=True); continue
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            position+=1; progress['next_position']=position; progress['successful_updates']+=1
            updates=progress['successful_updates']
            if updates==1 or updates%25==0:
                print(f'epoch={epoch+1}/{MAX_EPOCHS} position={position}/{len(order)} updates={updates} '
                      f'loss={loss.detach().item():.5f} ambiguous_words={masked}',flush=True)
            if updates%CHECKPOINT_EVERY==0:
                save_checkpoint(latest,model,optimizer,scheduler,scaler,progress,IDENTITY)
        per_label,macro,pred_count=validate(device)
        improved=macro>float(progress['best_val_macro_f1'])
        completed={**progress,'epoch':epoch+1,'next_position':0,'last_val_macro_f1':float(macro),
                   'epochs_without_improvement':0 if improved else progress['epochs_without_improvement']+1}
        if improved:
            completed['best_val_macro_f1']=float(macro); completed['best_epoch']=epoch+1
            save_checkpoint(best,model,optimizer,scheduler,scaler,completed,IDENTITY)
        progress=completed; save_checkpoint(latest,model,optimizer,scheduler,scaler,progress,IDENTITY)
        print(f'epoch={epoch+1} strict_macro_f1={macro:.6f} predictions={pred_count} improved={improved}',flush=True)
        print(per_label)
        if progress['epochs_without_improvement']>=PATIENCE:
            print('Early stopping after',PATIENCE,'epochs without improvement'); break
    print('Training finished:',progress,'elapsed_seconds',time.monotonic()-started)
else: print('Training skipped.')


AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
AMP overflow epoch=1 position=0; retrying
epoch=1/20 position=1/1281 updates=1 loss=1124.87231 ambiguous_words=0
epoch=1/20 position=25/1281 updates=25 loss=879.16479 ambiguous_words=0
epoch=1/20 position=50/1281 updates=50 loss=227.87939 ambiguous_words=0
epoch=1/20 position=75/1281 updates=75 loss=63.35217 ambiguous_words=0
epoch=1/20 position=100/1281 updates=100 loss=25.77783 ambiguous_words=0
epoch=1/20 position=125/1281 updates=125 loss=37.16821 ambiguous_words=0
epoch=1/20 position=150/1281 updates=150 loss=0.89128 ambiguous_words=11
epoch=1/20 position=175/1281 updates=175 loss=9.18481 ambiguous_words=0
epoch=1/20 position=200/1281 updates=200 loss

In [22]:
import gc, json, torch
for _name in ('best.pt', 'latest.pt'):
    _path = RUN_ROOT / _name
    print(_name, 'exists=', _path.is_file(), 'bytes=', _path.stat().st_size if _path.is_file() else None)
    if not _path.is_file():
        continue
    _ckpt = torch.load(_path, map_location='cpu', weights_only=False)
    print('top_keys=', sorted(_ckpt.keys()) if isinstance(_ckpt, dict) else type(_ckpt).__name__)
    _progress = _ckpt.get('progress', _ckpt.get('training_progress', {})) if isinstance(_ckpt, dict) else {}
    print('progress=', json.dumps(_progress, sort_keys=True, default=str))
    _state = next((_ckpt[k] for k in ('model_state_dict','model','state_dict') if isinstance(_ckpt, dict) and isinstance(_ckpt.get(k), dict)), None)
    if _state is not None:
        _head_keys = [k for k in _state if any(s in k.lower() for s in ('bilstm','emission','crf'))]
        print('state_tensor_count=', len(_state), 'task_head_key_count=', len(_head_keys), 'sample=', _head_keys[:5])
    del _ckpt
    gc.collect()
from pathlib import Path
import json
import zipfile
import torch

TEST_DIR = COURSE_ROOT / "Phase_2_LNER/Test"
BEST_CHECKPOINT = RUN_ROOT / "best.pt"
SUBMISSION_DIR = RUN_ROOT / "submission_best"
TEST_REPOSITORY = "Exploration-Lab/CS-781-Capstone"
TEST_REVISION = "89edba6a44d6bb9593fd13b6895b5cd3b06eb82e"
TEST_REPOSITORY_PATH = "Phase_2_LNER/Test/fold3-text.json"
test_files = [COURSE_ROOT / TEST_REPOSITORY_PATH]
missing_test_files = [path for path in test_files if not path.is_file()]
if missing_test_files:
    raise FileNotFoundError(f"Pinned Phase 2 NER test file is not readable locally: {missing_test_files}")
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Best BIO checkpoint not found: {BEST_CHECKPOINT}")
payload = torch.load(BEST_CHECKPOINT, map_location="cpu", weights_only=False)
if payload.get("schema_version") != 1 or payload.get("identity") != IDENTITY:
    raise ValueError("Best BIO checkpoint identity mismatch")
model.load_state_dict(payload["model"], strict=True)
test_rows = []
for path in test_files:
    test_rows.extend(load_records(path))
if not test_rows:
    raise ValueError("Unlabeled test files contained no documents; refusing empty submission")
if len({row["id"] for row in test_rows}) != len(test_rows):
    raise ValueError("Duplicate official document IDs across test files")
empty_text_ids = [row["id"] for row in test_rows if not row["text"]]
test_windows = [window for row in test_rows if row["text"] for window in make_document_windows(row, TOKENIZER, MAX_BPE, STRIDE_BPE)]
if not torch.cuda.is_available():
    raise RuntimeError("BIO test inference requires CUDA; refusing silent CPU fallback")
device = "cuda"
model.to(device).eval()
decoded = []
with torch.inference_mode():
    for window in test_windows:
        ids = torch.tensor([window["input_ids"]], dtype=torch.long, device=device)
        last = torch.tensor(window["word_last"], dtype=torch.long, device=device)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            emissions = model.emissions(ids, last)
        decoded.extend(decode_entities(window, model.crf.decode(emissions.float()), model.tags))
unique_predictions = sorted(set(decoded), key=lambda item: (item[0], item[1], item[2], item[3]))
scored_predictions = [(*item, 1.0) for item in unique_predictions]
texts = {row["id"]: row["text"] for row in test_rows}
prediction_path, zip_path = export_prediction([row["id"] for row in test_rows], scored_predictions, texts, SUBMISSION_DIR)
written = json.loads(prediction_path.read_text(encoding="utf-8"))
if list(written) != [row["id"] for row in test_rows]:
    raise ValueError("Export changed official test ID order or membership")
with zipfile.ZipFile(zip_path) as archive:
    if archive.namelist() != ["prediction.json"]:
        raise ValueError("Submission ZIP must contain only root prediction.json")
print("Loaded best checkpoint:", BEST_CHECKPOINT)
print("Best validation progress:", payload["progress"])
print("Test files:", [str(path) for path in test_files])
print("Documents:", len(test_rows), "decoded entities:", len(unique_predictions))
print("Empty official documents retained with []:", empty_text_ids)
print("Submission JSON:", prediction_path)
print("Submission ZIP:", zip_path)

best.pt exists= True bytes= 768084973
top_keys= ['cuda_rng', 'identity', 'model', 'optimizer', 'progress', 'python_rng', 'scaler', 'scheduler', 'schema_version', 'torch_rng']
progress= {"best_epoch": 6, "best_val_macro_f1": 0.44647299767734355, "epoch": 6, "epochs_without_improvement": 0, "last_val_macro_f1": 0.44647299767734355, "next_position": 0, "overflow_retries": 11, "successful_updates": 7686}
state_tensor_count= 126 task_head_key_count= 15 sample= ['bilstm.weight_ih_l0', 'bilstm.weight_hh_l0', 'bilstm.bias_ih_l0', 'bilstm.bias_hh_l0', 'bilstm.weight_ih_l0_reverse']
latest.pt exists= True bytes= 768086539
top_keys= ['cuda_rng', 'identity', 'model', 'optimizer', 'progress', 'python_rng', 'scaler', 'scheduler', 'schema_version', 'torch_rng']
progress= {"best_epoch": 6, "best_val_macro_f1": 0.44647299767734355, "epoch": 10, "epochs_without_improvement": 4, "last_val_macro_f1": 0.44484311776506735, "next_position": 0, "overflow_retries": 14, "successful_updates": 12810}
state_tensor

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, zipfile

source = Path('/content/drive/MyDrive/legal-llm-course/lorma_standalone_best_export.py')
original = source.read_text()
old = '    if identity.get(key)!=value: raise ValueError(f"LoRMA checkpoint identity mismatch: {key}")'
new = """    actual = identity.get(key)
    if key == 'model_config':
        try:
            actual = ModelConfig(**actual).validate().__dict__
            value = ModelConfig(**value).validate().__dict__
        except (TypeError, ValueError) as exc:
            raise ValueError('LoRMA checkpoint identity mismatch: model_config') from exc
    if actual!=value: raise ValueError(f"LoRMA checkpoint identity mismatch: {key}")"""
if original.count(old) != 1:
    raise RuntimeError(f'Expected exact identity check once; found {original.count(old)}')
patched = Path('/content/lorma_standalone_best_export_runtimefix.py')
patched.write_text(original.replace(old, new, 1))
print('Temporary exporter:', patched)
print('Mounted Drive source SHA256 (unchanged):', hashlib.sha256(original.encode()).hexdigest())
checkpoint = Path('/content/drive/MyDrive/legal-llm-course/runs/phase2_lner_multimethod/lorma_plus/gqa-prenorm-lorma-3ep-001-3bdf49acc8/best.pt')
outdir = checkpoint.parent / 'submission_best'
if (outdir / 'prediction.zip').exists() or (outdir / 'prediction.json').exists():
    raise FileExistsError(f'Refusing to overwrite existing LoRMA export: {outdir}')
proc = subprocess.Popen([sys.executable, str(patched)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for output_line in proc.stdout:
    print(output_line, end='', flush=True)
rc = proc.wait()
if rc:
    raise RuntimeError(f'Standalone LoRMA exporter exited with status {rc}')
zip_path = outdir / 'prediction.zip'
with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()
    if names != ['prediction.json']:
        raise AssertionError(f'Unexpected ZIP entries: {names}')
    payload = json.loads(zf.read('prediction.json'))
if len(payload) != 35 or payload.get('1305957') != []:
    raise AssertionError(f'Unexpected exported documents/empty ID: count={len(payload)}, empty={payload.get("1305957")!r}')
print('ZIP audit:', {'path': str(zip_path), 'bytes': zip_path.stat().st_size, 'entries': names, 'documents': len(payload), 'empty_1305957': payload['1305957']})

Temporary exporter: /content/lorma_standalone_best_export_runtimefix.py
Mounted Drive source SHA256 (unchanged): a70e52c51d6cb9e86d54a9362490c9a8e097132c598675f59cf49adfc96bdc90


In [8]:
# 8 · Artifact summary
if ACTION=='Train':
    print('Latest recovery:',RUN_ROOT/'latest.pt')
    print('Best validation checkpoint:',RUN_ROOT/'best.pt')
    print('All validation metrics used every original gold entity and exact character offsets.')
else: print('No training artifact written by this action.')


No training artifact written by this action.


In [9]:
# 9 · Deliberate next step
print('Test export is intentionally separate. First compare best validation macro-F1 and per-label recall.')


Test export is intentionally separate. First compare best validation macro-F1 and per-label recall.


## Running safely

1. Run all with `ACTION = "Inspect"` to verify the 64/6 split and ambiguity counts without loading a model.
2. Optionally run `ACTION = "Overfit check"` only when a GPU is free; it performs 20 updates on one window and writes no checkpoint.
3. Run `ACTION = "Train"` for full fine-tuning. Re-running resumes the preset-specific `latest.pt`; `best.pt` tracks strict validation macro-F1 with all original validation gold retained.
4. GQA is the default. Change only `PRESET` to `mha_pre` for the matched MHA experiment.


In [10]:
# Run immediately after the v1 Overfit check; do not rerun the 20 updates.
probe = train_windows[0]
probe_for_decode = dict(probe)
probe_for_decode["owner_start_index"] = probe_for_decode["word_start_index"]
probe_for_decode["owner_end_index"] = probe_for_decode["word_end_index"]

model_device = next(model.parameters()).device
was_training = model.training
model.eval()
with torch.inference_mode():
    ids = torch.tensor(
        [probe_for_decode["input_ids"]], dtype=torch.long, device=model_device
    )
    word_last = torch.tensor(
        probe_for_decode["word_last"], dtype=torch.long, device=model_device
    )
    with torch.autocast(
        device_type=model_device.type,
        dtype=torch.float16,
        enabled=(model_device.type == "cuda"),
    ):
        emissions = model.emissions(ids, word_last)
    predicted = set(
        decode_entities(
            probe_for_decode,
            model.crf.decode(emissions.float()),
            model.tags,
        )
    )
model.train(was_training)

left = probe_for_decode["word_spans"][0][0]
right = probe_for_decode["word_spans"][-1][1]
document_id = probe_for_decode["document_id"]
gold = {
    (document_id, start, end, label)
    for start, end, label in probe_for_decode["entities"]
    if left <= start and end <= right
}

tp = len(predicted & gold)
fp = len(predicted - gold)
fn = len(gold - predicted)
precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print(
    {
        "document_id": document_id,
        "window_chars": [left, right],
        "gold": len(gold),
        "predicted": len(predicted),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "strict_micro_f1": f1,
    }
)

print("\nGOLD")
for _, start, end, label in sorted(gold):
    print(label, [start, end], repr(probe_for_decode["text"][start:end]))

print("\nPREDICTED")
for _, start, end, label in sorted(predicted):
    marker = "TP" if (document_id, start, end, label) in gold else "FP"
    print(marker, label, [start, end], repr(probe_for_decode["text"][start:end]))

print("\nMISSED GOLD")
for _, start, end, label in sorted(gold - predicted):
    print(label, [start, end], repr(probe_for_decode["text"][start:end]))


{'document_id': '115651329', 'window_chars': [0, 1650], 'gold': 12, 'predicted': 0, 'tp': 0, 'fp': 0, 'fn': 12, 'precision': 0.0, 'recall': 0.0, 'strict_micro_f1': 0.0}

GOLD
RESP [137, 153] 'STATE OF HARYANA'
DATE [252, 261] '19.8.2011'
COURT [276, 308] 'High Court of Punjab and Haryana'
COURT [359, 369] 'High Court'
DATE [575, 585] '18.11.1994'
WIT [637, 644] 'Jagdish'
WIT [695, 704] 'Mange Ram'
APP [792, 800] 'Jage Ram'
APP [1122, 1130] 'Jage Ram'
APP [1341, 1349] 'Jage Ram'
WIT [1373, 1380] 'Jagdish'
WIT [1583, 1590] 'Jagdish'

PREDICTED

MISSED GOLD
RESP [137, 153] 'STATE OF HARYANA'
DATE [252, 261] '19.8.2011'
COURT [276, 308] 'High Court of Punjab and Haryana'
COURT [359, 369] 'High Court'
DATE [575, 585] '18.11.1994'
WIT [637, 644] 'Jagdish'
WIT [695, 704] 'Mange Ram'
APP [792, 800] 'Jage Ram'
APP [1122, 1130] 'Jage Ram'
APP [1341, 1349] 'Jage Ram'
WIT [1373, 1380] 'Jagdish'
WIT [1583, 1590] 'Jagdish'


In [11]:
# Read-only post-overfit inspection: no model calls, gradients, optimizer steps, or writes.
import inspect, collections
print('BIO label globals:', {k:repr(v)[:400] for k,v in globals().items() if ('TAG' in k.upper() or 'LABEL' in k.upper()) and isinstance(v,(list,tuple,dict))})
matching=[w for w in train_windows if str(w.get('document_id'))=='115651329']
print('matching windows:',len(matching),'keys:',sorted(matching[0].keys()) if matching else None)
if matching:
    w=next((x for x in matching if x.get('char_start',x.get('start_char',x.get('char_range',(None,None))[0] if isinstance(x.get('char_range'),(tuple,list)) else None))==0),matching[0])
    print('selected_window metadata:', {k:repr(v)[:200] for k,v in w.items() if k not in ('input_ids','offsets','word_ids','entities','labels')})
    print('targets_and_segments source:',inspect.getsource(targets_and_segments))
    targets,segments,masked=targets_and_segments(w,LABELS)
    print('target_type/shape:',type(targets).__name__,getattr(targets,'shape',None),'segments',len(segments),'masked',len(masked))
    vals=targets.tolist() if hasattr(targets,'tolist') else list(targets)
    flat=[int(v) for v in vals]
    non_o=[v for v in flat if v!=0]
    print('BIO target tag counts:',collections.Counter(flat),'non_O_count:',len(non_o),'total_positions:',len(flat))
print('model_present:', 'model' in globals())
if 'model' in globals():
    named=list(model.named_parameters())
    head=[(n,p) for n,p in named if not n.startswith('backbone.')]
    head_grads=[(n,p) for n,p in head if p.grad is not None]
    print('head_parameter_count:',len(head),'head_grad_tensors:',len(head_grads),'head_grad_params:',sum(p.grad is not None for n,p in head))
    print('head_gradient_summary:',[(n,tuple(p.grad.shape),bool(torch.isfinite(p.grad).all().item()),float(p.grad.norm().item())) for n,p in head_grads[:30]])


BIO label globals: {'LABELS': "('APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO')"}
matching windows: 13 keys: ['document_id', 'entities', 'input_ids', 'owner_end_index', 'owner_start_index', 'text', 'word_end_index', 'word_last', 'word_spans', 'word_start_index']
selected_window metadata: {'document_id': "'115651329'", 'text': "'REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J.", 'word_spans': '[(0, 10), (11, 13), (14, 17), (18, 25), (26, 31), (32, 34), (35, 40), (41, 49), (50, 59), (60, 72), (73, 81), (82, 88), (89, 91), (91, 92), (93, 95), (95, 96), (96, 100), (101, 105), (106, 109), (110,', 'word_last': '[5, 6, 9, 14, 18, 20, 24, 29, 36, 45, 50, 55, 56, 57, 58, 59, 62, 66, 69, 70, 72, 73, 74, 75, 79, 81, 85, 87, 92, 93, 94, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 1

In [12]:
from collections import Counter

device = "cuda"
probe = train_windows[0]
targets, segments, ambiguous = targets_and_segments(probe, LABELS)
print("target tags:", Counter(model.tags[index] for index in targets.tolist()))
print("segments:", segments, "ambiguous:", ambiguous)

ids = torch.tensor([probe["input_ids"]], dtype=torch.long, device=device)
word_last = torch.tensor(probe["word_last"], dtype=torch.long, device=device)
targets = targets.to(device)

model.eval()
with torch.no_grad():
    hidden = model.backbone_hidden(ids)
    word_features = hidden[:, word_last, :].detach().float()

for parameter in model.backbone.parameters():
    parameter.requires_grad_(False)

head_parameters = (list(model.bilstm.parameters()) + list(model.emission.parameters()) + list(model.crf.parameters()))
head_dtypes = {parameter.dtype for parameter in head_parameters}
if head_dtypes != {torch.float32}:
    raise TypeError(f"expected FP32 task head; found {head_dtypes}")
if word_features.dtype != torch.float32:
    raise TypeError(f"expected cached FP32 word features; found {word_features.dtype}")
print("cached feature dtype:", word_features.dtype, "head dtype:", next(iter(head_dtypes)))

probe_optimizer = torch.optim.AdamW(head_parameters, lr=3e-3, weight_decay=0.0)
probe_for_decode = dict(probe)
probe_for_decode["owner_start_index"] = probe["word_start_index"]
probe_for_decode["owner_end_index"] = probe["word_end_index"]
left = probe["word_spans"][0][0]
right = probe["word_spans"][-1][1]
gold = {(probe["document_id"], start, end, label) for start, end, label in probe["entities"] if left <= start and end <= right}

def head_emissions():
    contextual, _ = model.bilstm(model.dropout(word_features))
    return model.emission(model.dropout(contextual))[0]

def decoded_metrics(emissions):
    predicted = set(decode_entities(probe_for_decode, model.crf.decode(emissions.float()), model.tags))
    tp = len(predicted & gold)
    fp = len(predicted - gold)
    fn = len(gold - predicted)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"gold": len(gold), "predicted": len(predicted), "tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "strict_micro_f1": f1}

model.train()
for update in range(1, 201):
    probe_optimizer.zero_grad(set_to_none=True)
    emissions = head_emissions()
    losses = [model.crf.neg_log_likelihood(emissions[first:end], targets[first:end]) for first, end in segments]
    loss = torch.stack(losses).mean()
    loss.backward()
    if update == 1:
        print("first-update gradient norms:", {
            "bilstm": float(torch.nn.utils.clip_grad_norm_(model.bilstm.parameters(), float("inf"))),
            "emission": float(torch.nn.utils.clip_grad_norm_(model.emission.parameters(), float("inf"))),
            "crf": float(torch.nn.utils.clip_grad_norm_(model.crf.parameters(), float("inf"))),
        })
    torch.nn.utils.clip_grad_norm_(head_parameters, 5.0)
    probe_optimizer.step()
    if update % 20 == 0:
        model.eval()
        with torch.no_grad():
            metrics = decoded_metrics(head_emissions())
        model.train()
        print("update", update, "loss", float(loss.detach()), metrics, flush=True)
        if metrics["tp"] == len(gold) and metrics["fp"] == 0 and metrics["fn"] == 0:
            print("Exact probe recovery reached; stopping early at update", update)
            break
print("Head-only diagnostic complete; no checkpoint was written.")

target tags: Counter({'O': 333, 'I-DATE': 8, 'I-COURT': 6, 'B-WIT': 4, 'B-APP': 3, 'I-APP': 3, 'I-RESP': 2, 'B-DATE': 2, 'B-COURT': 2, 'B-RESP': 1, 'I-WIT': 1})
segments: [(0, 365)] ambiguous: []
cached feature dtype: torch.float32 head dtype: torch.float32
first-update gradient norms: {'bilstm': 125.12672424316406, 'emission': 166.50550842285156, 'crf': 25.95263671875}
update 20 loss 14.318359375 {'gold': 12, 'predicted': 11, 'tp': 10, 'fp': 1, 'fn': 2, 'precision': 0.9090909090909091, 'recall': 0.8333333333333334, 'strict_micro_f1': 0.8695652173913043}
update 40 loss 1.240966796875 {'gold': 12, 'predicted': 12, 'tp': 12, 'fp': 0, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'strict_micro_f1': 1.0}
Exact probe recovery reached; stopping early at update 40
Head-only diagnostic complete; no checkpoint was written.


In [16]:
BIO_HEAD_PROBE_MODIFIED = True
BIO_HEAD_PROBE_STOPPED_AT_UPDATE = 40
model._bio_head_probe_modified = True
print({"probe_modified": BIO_HEAD_PROBE_MODIFIED, "stop_update": BIO_HEAD_PROBE_STOPPED_AT_UPDATE, "backbone_frozen": all(not p.requires_grad for p in model.backbone.parameters())})
print("Before any full training, rebuild the model cleanly and restore backbone requires_grad.")

{'probe_modified': True, 'stop_update': 40, 'backbone_frozen': True}
Before any full training, rebuild the model cleanly and restore backbone requires_grad.
